In [1]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.patches as mpatches

In [2]:
# SET RANDOM SEED

np.random.seed(42)
rows = 80

In [3]:
#  CREATE SYNTHETIC DATAFRAME

df = pd.DataFrame({
    "model_series": np.random.choice([220, 240, 260], rows),
    "model_sku": np.random.choice(["A1L", "A2L", "A1M", "A2M", "A2H"], rows),
    "cpu_util_avg": np.round(np.random.uniform(40, 95, rows), 1),
    "memory_util_avg": np.round(np.random.uniform(45, 95, rows), 1),
    "port_util_avg": np.round(np.random.uniform(25, 95, rows), 1),
    "interface_error_rate": np.round(np.random.uniform(0, 6, rows), 2),
    "connection_rate_cps": np.random.randint(10000, 250000, rows),
    "syn_flood_events": np.random.randint(0, 60, rows),
    "inspection_drop_rate": np.round(np.random.uniform(0, 10, rows), 2),
    "fragmented_packet_pct": np.round(np.random.uniform(0, 20, rows), 2),
    "ssl_decrypt_enabled": np.random.choice([0, 1], rows),
    "image_upgrade_count": np.random.randint(0, 6, rows),
    "reboot_count": np.random.randint(0, 6, rows),
    "failover_events": np.random.randint(0, 4, rows)
})

In [4]:
#  CORE FILE LOGIC

df["core_file_count"] = (
    (df["cpu_util_avg"] > 78).astype(int) +
    (df["memory_util_avg"] > 82).astype(int) +
    (df["interface_error_rate"] > 2.5).astype(int) +
    (df["inspection_drop_rate"] > 4).astype(int) +
    (df["syn_flood_events"] > 25).astype(int) +
    (df["ssl_decrypt_enabled"] == 1).astype(int) +
    np.random.poisson(0.5, rows)   # smaller Poisson for realistic distribution
)

df["core_file_flag"] = (df["core_file_count"] > 2).astype(int)  # increase threshold to reduce red bar

In [5]:
# DEVICE STABILITY SCORE

def stability_score(row):
    score = 100
    score -= row["cpu_util_avg"] * 0.22
    score -= row["memory_util_avg"] * 0.22
    score -= row["interface_error_rate"] * 4
    score -= row["port_util_avg"] * 0.14
    score -= row["core_file_count"] * 8
    score -= row["reboot_count"] * 3
    score -= row["image_upgrade_count"] * 2
    score -= row["failover_events"] * 4
    return max(0, min(100, round(score, 1)))

df["device_stability_score"] = df.apply(stability_score, axis=1)

In [6]:
# SEVERITY LEVEL LOGIC

def severity_level(score):
    if score >= 75:
        return "Low"
    elif score >= 50:
        return "Medium"
    else:
        return "High"

df["severity_level"] = df["device_stability_score"].apply(severity_level)

In [7]:
# SAVE RAW DATASET

raw_file = "cisco_firewall_final_dataset_with_severity.csv"
df.to_csv(raw_file, index=False)
raw_file
df

,model_series,model_sku,cpu_util_avg,memory_util_avg,port_util_avg,interface_error_rate,connection_rate_cps,syn_flood_events,inspection_drop_rate,fragmented_packet_pct,ssl_decrypt_enabled,image_upgrade_count,reboot_count,failover_events,core_file_count,core_file_flag,device_stability_score,severity_level
0,260,A2M,46.7,87.5,87.1,3.44,64693,53,1.34,17.88,1,2,3,2,5,1,0.0,High
1,220,A1L,59.6,91.8,26.9,0.77,36962,2,0.14,2.57,0,0,3,1,1,0,38.8,High
2,260,A2H,89.9,84.3,65.5,4.87,38295,15,0.75,6.60,0,4,1,3,3,1,0.0,High
3,260,A2H,55.0,78.4,55.7,4.92,23807,22,6.92,6.43,1,5,2,0,4,1,0.0,High
4,220,A2L,75.6,74.0,72.0,3.76,183420,56,5.34,1.85,0,3,2,2,3,1,0.0,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,240,A2M,78.5,76.4,91.1,1.34,48413,12,3.48,9.81,1,4,0,2,2,0,15.8,High
76,240,A2M,86.6,54.7,85.7,2.38,62638,32,0.32,14.46,1,1,2,3,3,1,3.4,High
77,260,A2M,87.1,48.5,69.5,5.35,193808,33,5.49,16.42,1,2,2,1,5,1,0.0,High
78,260,A1M,62.2,64.8,81.1,0.88,248964,40,5.34,14.37,0,2,3,1,2,0,24.2,High
